# Render-and-Detect for Word Stroke Detection

**Approach**: Render strokes to a binary image, detect word regions using connected components on the rendered image, then map each connected component back to the strokes that produced it.

This is a reference baseline that tests the idea of working in the **offline** (rendered image) domain — without needing a trained neural network. It measures how well a simple render→segment→map-back pipeline performs at the stroke-grouping task.

**Parameters to tune** (via LOPO CV):
- `stroke_width`: rendering line thickness (controls how close strokes need to be to merge into the same component)
- `scale`: image resolution scale factor

**Evaluation**: Leave-one-page-out cross-validation (6 folds), same metrics as the baseline experiments.

## 1. Data Loading

In [ ]:
import gzip
import json
import warnings
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from bs4 import BeautifulSoup as bs
from bs4 import XMLParsedAsHTMLWarning
from huggingface_hub import snapshot_download
from PIL import Image, ImageDraw
from tabulate import tabulate

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

_TEXT_CLASSES = {"word", "digit", "mathematical_expression"}


@dataclass
class Stroke:
    x: np.ndarray
    y: np.ndarray


@dataclass
class PageData:
    page_id: str
    strokes: list[Stroke]
    word_groups: list[list[int]]


def _parse_strokes_per_layer(path: Path) -> list[list[list[Stroke]]]:
    with gzip.open(path, "r") as f:
        content = f.read().decode("utf-8")
    soup = bs(content, "lxml")
    pages = []
    for page in soup.find_all("page"):
        layers = []
        for layer in page.find_all("layer"):
            layer_strokes = []
            for stroke_el in layer.find_all("stroke"):
                coords = np.fromstring(stroke_el.text, sep=" ").reshape(-1, 2).T
                layer_strokes.append(Stroke(x=coords[0], y=coords[1]))
            layers.append(layer_strokes)
        pages.append(layers)
    return pages


def load_benchmark_pages() -> list[PageData]:
    local_dir = Path(
        snapshot_download("PellelNitram/xournalpp_htr_benchmark", repo_type="dataset")
    )
    data_dir = local_dir / "data"
    xopp_files = sorted(data_dir.glob("*.xopp")) + sorted(data_dir.glob("*.xoj"))

    pages = []
    for xopp_path in xopp_files:
        gt_path = xopp_path.with_suffix("").with_suffix(".gt.json")
        if not gt_path.exists():
            continue
        strokes_per_layer = _parse_strokes_per_layer(xopp_path)
        with open(gt_path) as f:
            gt = json.load(f)

        page_annotations: dict[int, list[list[int]]] = {}
        for ann in gt["annotations"]:
            if ann["class"] not in _TEXT_CLASSES:
                continue
            page_idx = ann["page_index"]
            layer_idx = ann["layer_index"]
            offset = sum(
                len(strokes_per_layer[page_idx][li]) for li in range(layer_idx)
            )
            global_indices = [offset + si for si in ann["stroke_indices"]]
            page_annotations.setdefault(page_idx, []).append(global_indices)

        for page_idx, layer_list in enumerate(strokes_per_layer):
            flat_strokes = [s for layer in layer_list for s in layer]
            if not flat_strokes:
                continue
            word_groups = page_annotations.get(page_idx, [])
            page_id = f"{xopp_path.stem}_p{page_idx}"
            pages.append(
                PageData(page_id=page_id, strokes=flat_strokes, word_groups=word_groups)
            )

    return pages

In [ ]:
pages = load_benchmark_pages()
for p in pages:
    n_word_strokes = len({si for wg in p.word_groups for si in wg})
    print(
        f"{p.page_id}: {len(p.strokes)} strokes, {len(p.word_groups)} words, {n_word_strokes} word strokes"
    )

## 2. Stroke Rendering

Render strokes onto a binary image using PIL. Each stroke is drawn individually so we can track which stroke index maps to which pixels.

In [ ]:
def render_strokes(
    strokes: list[Stroke],
    scale: float = 1.0,
    stroke_width: int = 3,
    margin: int = 10,
) -> tuple[np.ndarray, np.ndarray, float, float]:
    """Render strokes to a binary image and a per-stroke label map.

    Returns:
        combined_img: binary image (0=background, 255=foreground)
        stroke_label_map: int array where pixel value = stroke_index+1 (0=background).
                          When strokes overlap, last-drawn wins.
        offset_x, offset_y: coordinate offsets applied
    """
    all_x = np.concatenate([s.x for s in strokes])
    all_y = np.concatenate([s.y for s in strokes])
    min_x, max_x = all_x.min(), all_x.max()
    min_y, max_y = all_y.min(), all_y.max()

    w = int((max_x - min_x) * scale) + 2 * margin + 1
    h = int((max_y - min_y) * scale) + 2 * margin + 1

    combined = Image.new("L", (w, h), 255)
    combined_draw = ImageDraw.Draw(combined)

    stroke_label_map = np.zeros((h, w), dtype=np.int32)

    for si, stroke in enumerate(strokes):
        xs = (stroke.x - min_x) * scale + margin
        ys = (stroke.y - min_y) * scale + margin
        coords = list(zip(xs.tolist(), ys.tolist(), strict=False))

        if len(coords) < 2:
            ix, iy = int(round(coords[0][0])), int(round(coords[0][1]))
            if 0 <= ix < w and 0 <= iy < h:
                stroke_label_map[iy, ix] = si + 1
                combined.putpixel((ix, iy), 0)
            continue

        combined_draw.line(coords, fill=0, width=stroke_width)

        single = Image.new("L", (w, h), 255)
        single_draw = ImageDraw.Draw(single)
        single_draw.line(coords, fill=0, width=stroke_width)
        mask = np.array(single) < 128
        stroke_label_map[mask] = si + 1

    combined_img = np.array(combined)
    return (
        combined_img,
        stroke_label_map,
        min_x - margin / scale,
        min_y - margin / scale,
    )

## 3. Word Region Detection

Use OpenCV connected components on the binarized rendered image to find word regions.

In [ ]:
def detect_regions(binary_img: np.ndarray) -> np.ndarray:
    """Find connected components on a binary image.

    Returns a label map where each connected foreground region has a unique integer label.
    Background is 0.
    """
    fg = (binary_img < 128).astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(fg, connectivity=8)
    return labels

## 4. Stroke-to-Region Assignment

Map each stroke to the connected component region it overlaps most with, using the per-stroke label map.

In [ ]:
def assign_strokes_to_regions(
    stroke_label_map: np.ndarray,
    region_labels: np.ndarray,
    n_strokes: int,
) -> list[set[int]]:
    """Assign each stroke to its dominant connected-component region.

    For each stroke, look at the pixels it occupies (from stroke_label_map)
    and find which region label appears most often. Group strokes by region.
    """
    stroke_to_region = {}

    for si in range(n_strokes):
        mask = stroke_label_map == (si + 1)
        if not mask.any():
            continue
        region_vals = region_labels[mask]
        region_vals = region_vals[region_vals > 0]
        if len(region_vals) == 0:
            continue
        dominant_region = int(np.bincount(region_vals).argmax())
        stroke_to_region[si] = dominant_region

    region_to_strokes: dict[int, set[int]] = {}
    for si, region in stroke_to_region.items():
        region_to_strokes.setdefault(region, set()).add(si)

    return list(region_to_strokes.values())


def render_detect_cluster(
    strokes: list[Stroke],
    scale: float = 1.0,
    stroke_width: int = 3,
) -> tuple[list[set[int]], np.ndarray, np.ndarray]:
    """Full pipeline: render → connected components → assign strokes."""
    combined_img, stroke_label_map, _, _ = render_strokes(strokes, scale, stroke_width)
    region_labels = detect_regions(combined_img)
    clusters = assign_strokes_to_regions(stroke_label_map, region_labels, len(strokes))
    return clusters, combined_img, region_labels

## 5. Evaluation

In [ ]:
@dataclass
class GroupingMetrics:
    precision: float
    recall: float
    f1: float
    stroke_accuracy: float
    n_gt_words: int
    n_pred_clusters: int


def evaluate_grouping(
    predicted_clusters: list[set[int]],
    gt_word_groups: list[list[int]],
    n_strokes: int,
    jaccard_threshold: float = 0.5,
) -> GroupingMetrics:
    gt_sets = [set(wg) for wg in gt_word_groups]

    candidates = []
    for i, pred in enumerate(predicted_clusters):
        for j, gt in enumerate(gt_sets):
            intersection = len(pred & gt)
            union = len(pred | gt)
            if union > 0:
                jaccard = intersection / union
                if jaccard >= jaccard_threshold:
                    candidates.append((jaccard, i, j))

    candidates.sort(key=lambda x: x[0], reverse=True)

    matched_pred = set()
    matched_gt = set()
    for _, pi, gi in candidates:
        if pi not in matched_pred and gi not in matched_gt:
            matched_pred.add(pi)
            matched_gt.add(gi)

    n_matched = len(matched_pred)
    n_pred = len(predicted_clusters)
    n_gt = len(gt_sets)

    precision = n_matched / n_pred if n_pred > 0 else 0.0
    recall = n_matched / n_gt if n_gt > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    stroke_to_gt = {}
    for gi, gt in enumerate(gt_sets):
        for si in gt:
            stroke_to_gt[si] = gi

    stroke_to_pred = {}
    for pi, pred in enumerate(predicted_clusters):
        for si in pred:
            stroke_to_pred[si] = pi

    correct = 0
    total_word_strokes = len(stroke_to_gt)
    for si, gi in stroke_to_gt.items():
        if si in stroke_to_pred:
            pi = stroke_to_pred[si]
            pred_set = predicted_clusters[pi]
            gt_set = gt_sets[gi]
            intersection = len(pred_set & gt_set)
            union = len(pred_set | gt_set)
            if union > 0 and intersection / union >= jaccard_threshold:
                correct += 1

    stroke_accuracy = correct / total_word_strokes if total_word_strokes > 0 else 0.0

    return GroupingMetrics(
        precision=precision,
        recall=recall,
        f1=f1,
        stroke_accuracy=stroke_accuracy,
        n_gt_words=n_gt,
        n_pred_clusters=n_pred,
    )

## 6. LOPO Cross-Validation

Tune `stroke_width` on the 5 training pages per fold (grid search over candidate widths at a fixed scale), evaluate on the held-out page.

In [ ]:
SCALE = 0.5  # fixed rendering scale


def tune_stroke_width(
    train_pages: list[PageData],
    scale: float = SCALE,
    candidates: list[int] | None = None,
) -> int:
    """Find stroke_width that maximises mean F1 on training pages."""
    if candidates is None:
        candidates = list(range(1, 16))

    best_width = candidates[0]
    best_f1 = -1.0

    for width in candidates:
        f1s = []
        for page in train_pages:
            clusters, _, _ = render_detect_cluster(page.strokes, scale, width)
            metrics = evaluate_grouping(clusters, page.word_groups, len(page.strokes))
            f1s.append(metrics.f1)
        mean_f1 = np.mean(f1s)
        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_width = width

    return best_width


results = []
for i, test_page in enumerate(pages):
    train_pages = pages[:i] + pages[i + 1 :]
    width = tune_stroke_width(train_pages)
    clusters, rendered, region_labels = render_detect_cluster(
        test_page.strokes, SCALE, width
    )
    metrics = evaluate_grouping(clusters, test_page.word_groups, len(test_page.strokes))
    results.append(
        (test_page.page_id, width, metrics, clusters, rendered, region_labels)
    )
    print(
        f"Fold {i+1}/6: {test_page.page_id} — stroke_width={width}, F1={metrics.f1:.3f}"
    )

## 7. Results

In [ ]:
headers = [
    "Page",
    "Stroke Width",
    "Precision",
    "Recall",
    "F1",
    "Stroke Acc",
    "GT Words",
    "Pred Clusters",
]
rows = []
for page_id, width, m, _, _, _ in results:
    rows.append(
        [
            page_id,
            width,
            f"{m.precision:.3f}",
            f"{m.recall:.3f}",
            f"{m.f1:.3f}",
            f"{m.stroke_accuracy:.3f}",
            m.n_gt_words,
            m.n_pred_clusters,
        ]
    )

precisions = [m.precision for _, _, m, _, _, _ in results]
recalls = [m.recall for _, _, m, _, _, _ in results]
f1s = [m.f1 for _, _, m, _, _, _ in results]
stroke_accs = [m.stroke_accuracy for _, _, m, _, _, _ in results]

rows.append(
    [
        "MEAN",
        "",
        f"{np.mean(precisions):.3f} ± {np.std(precisions):.3f}",
        f"{np.mean(recalls):.3f} ± {np.std(recalls):.3f}",
        f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
        f"{np.mean(stroke_accs):.3f} ± {np.std(stroke_accs):.3f}",
        "",
        "",
    ]
)

print(tabulate(rows, headers=headers, tablefmt="grid"))

print("\n\nComparison Summary:")
print(
    tabulate(
        [
            [
                "Spatial Baseline",
                "0.762 ± 0.175",
                "0.734 ± 0.213",
                "0.733 ± 0.178",
                "0.641 ± 0.259",
            ],
            [
                "Gradient Boosting",
                "0.738 ± 0.219",
                "0.741 ± 0.318",
                "0.719 ± 0.288",
                "0.768 ± 0.286",
            ],
            [
                "Render + CC",
                f"{np.mean(precisions):.3f} ± {np.std(precisions):.3f}",
                f"{np.mean(recalls):.3f} ± {np.std(recalls):.3f}",
                f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
                f"{np.mean(stroke_accs):.3f} ± {np.std(stroke_accs):.3f}",
            ],
        ],
        headers=["Method", "Precision", "Recall", "F1", "Stroke Acc"],
        tablefmt="grid",
    )
)

## 8. Visualize

Show rendered image with detected regions and GT vs predicted stroke groups side by side.

In [ ]:
def plot_render_detect(
    page: PageData,
    clusters: list[set[int]],
    rendered: np.ndarray,
    region_labels: np.ndarray,
    title: str = "",
):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    cmap = plt.colormaps["tab20"]

    # 1. Rendered image + region overlay
    axes[0].set_title("Rendered + Regions")
    axes[0].imshow(rendered, cmap="gray")
    region_overlay = np.zeros((*region_labels.shape, 4))
    unique_labels = np.unique(region_labels)
    for label in unique_labels:
        if label == 0:
            continue
        mask = region_labels == label
        color = cmap(label % 20)
        region_overlay[mask] = (*color[:3], 0.4)
    axes[0].imshow(region_overlay)
    axes[0].axis("off")

    # 2. GT stroke groups
    axes[1].set_title("Ground Truth")
    word_stroke_set = {si for wg in page.word_groups for si in wg}
    for gi, wg in enumerate(page.word_groups):
        color = cmap(gi % 20)
        for si in wg:
            s = page.strokes[si]
            axes[1].plot(s.x, -s.y, color=color, linewidth=1.5)
    for si, s in enumerate(page.strokes):
        if si not in word_stroke_set:
            axes[1].plot(s.x, -s.y, color="lightgray", linewidth=0.5)
    axes[1].set_aspect("equal")
    axes[1].axis("off")

    # 3. Predicted clusters
    axes[2].set_title("Predicted Clusters")
    clustered_strokes = {si for c in clusters for si in c}
    for ci, cluster in enumerate(clusters):
        color = cmap(ci % 20)
        for si in cluster:
            s = page.strokes[si]
            axes[2].plot(s.x, -s.y, color=color, linewidth=1.5)
    for si, s in enumerate(page.strokes):
        if si not in clustered_strokes:
            axes[2].plot(s.x, -s.y, color="lightgray", linewidth=0.5)
    axes[2].set_aspect("equal")
    axes[2].axis("off")

    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


for page_id, width, m, clusters, rendered, region_labels in results:
    page = next(p for p in pages if p.page_id == page_id)
    plot_render_detect(
        page,
        clusters,
        rendered,
        region_labels,
        f"{page_id} (stroke_width={width}, F1={m.f1:.3f})",
    )